# RQ1 — Decomposition + robustness on the two dose-controlled schemes (v2 + v3)

This notebook is a **subset-of-schemes** re-estimate of the analysis in
[rq1_directional_reframe.ipynb](rq1_directional_reframe.ipynb). The
coh ≥ 30 filter is unchanged; the only change is dropping the **normFalse**
(v1, Phase 12) rows.

## Why the schemes are subset, not the data

`normFalse` is the naive-additive injection δ = α·(v̂_a + v̂_b). Its total ‖δ‖
scales as α·√(2(1+cos)), so cosine is **mechanically confounded** with the
delivered dose (pooled r(cos, ‖δ‖) ≈ +0.995 within the v1 frame). This is the
scheme we explicitly debunked in the audit notebook — the |cos|→regime signal
on Phase 12 turned out to be a coherence-collapse artefact downstream of that
dose confound.

The dose-controlled schemes are:
- **`normTrue` (v2 / Phase 12.5)** — δ = α·(v̂_a + v̂_b)/‖v̂_a + v̂_b‖; total
  ‖δ‖ = α exactly.
- **`per_axis` (v3 / Phase 15.16)** — per-axis push fixed at α; total
  ‖δ‖ = α·√(2/(1+cos)).

So the geometric effect can be estimated on these two without the cos↔dose
confound bleeding into the cos coefficient. This is the *principled* sample,
not the *flattering* sample — we report whether dropping normFalse strengthens
or weakens the result regardless.

## Identifiability flag (read before interpreting)

With only two schemes:
- `mechanical_push` under normTrue is α·√((1+cos)/2) — a monotone function of
  cos *within scheme*.
- `mechanical_push` under per_axis is constant α (no within-scheme variance).

So in a two-scheme pooled fit, the only between-scheme variance in
mechanical_push is captured by the scheme dummy, and the within-scheme variance
is collinear with cos. We therefore expect the VIF for cos in the
`{mean_single, max_single, mechanical_push, cos}` design to be **large**, and
the standalone β_cos in that model to be poorly identified. The notebook reports
the VIF and a **sensitivity without mechanical_push** explicitly. The
sensitivity is the more trustworthy reading when VIF blows up.

## Parameterisation note

Two schemes, one dummy: `d_per_axis ∈ {0, 1}`, with `normTrue` as the
reference. Using two dummies + intercept would be exactly collinear (the
two-dummy design we used at n=76 needed all three schemes to be identifiable).

In [1]:
# Imports + paths + rng + utilities (matching the directional notebook)
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

REPO = Path.cwd().parents[1]
RNG  = np.random.default_rng(0)

CSV_PATH = REPO / "analysis/rq1_consolidated/consolidated_coh30.csv"
df_all = pd.read_csv(CSV_PATH)
df_all["supp_signed_a"]    = df_all["delta_a_joint"] - df_all["delta_a_single"]
df_all["supp_signed_b"]    = df_all["delta_b_joint"] - df_all["delta_b_single"]
df_all["supp_mean_signed"] = 0.5 * (df_all["supp_signed_a"] + df_all["supp_signed_b"])
print(f"Full frame (n=76 reference):  schemes = {dict(df_all['scheme'].value_counts())}")

# v2 + v3 subset
df = df_all[df_all['scheme'].isin(['normTrue', 'per_axis'])].reset_index(drop=True)
print(f"\nv2+v3 frame: n = {len(df)}  schemes = {dict(df['scheme'].value_counts())}")
print(f"  pairs containing apathetic or hallucinating: "
      f"{((df.trait_a.isin(['apathetic','hallucinating'])) | (df.trait_b.isin(['apathetic','hallucinating']))).sum()}/{len(df)}")

Full frame (n=76 reference):  schemes = {'normTrue': 28, 'normFalse': 26, 'per_axis': 22}

v2+v3 frame: n = 50  schemes = {'normTrue': 28, 'per_axis': 22}
  pairs containing apathetic or hallucinating: 23/50


In [2]:
# OLS utilities — same as previous notebooks, but with single-dummy 2-scheme design
def fit_ols(X, y):
    X = np.asarray(X, float); y = np.asarray(y, float)
    n, k = X.shape
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    rss = float(resid @ resid)
    dof = n - k
    sigma2 = rss / dof if dof > 0 else np.nan
    XtX_inv = np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    tss = float(((y - y.mean())**2).sum())
    r2 = 1 - rss / tss if tss > 0 else np.nan
    return {"beta": beta, "se": se, "rss": rss, "n": n, "k": k,
            "dof": dof, "r2": r2}


def pooled_design_2sch(sub, cols):
    """Two-scheme design: single dummy d_per_axis, normTrue reference."""
    Z = (StandardScaler().fit_transform(sub[cols].values)
         if cols else np.empty((len(sub), 0)))
    d_pa = (sub["scheme"] == "per_axis").astype(float).values
    X = np.column_stack([np.ones(len(sub)), Z, d_pa])
    names = ["(intercept)"] + list(cols) + ["d_per_axis"]
    return X, names


def pooled_design_3sch(sub, cols):
    """Three-scheme design: two dummies (normFalse reference) — matches the directional notebook."""
    Z = (StandardScaler().fit_transform(sub[cols].values)
         if cols else np.empty((len(sub), 0)))
    d_nT = (sub["scheme"] == "normTrue").astype(float).values
    d_pa = (sub["scheme"] == "per_axis").astype(float).values
    X = np.column_stack([np.ones(len(sub)), Z, d_nT, d_pa])
    names = ["(intercept)"] + list(cols) + ["d_normTrue", "d_per_axis"]
    return X, names


def fit_pooled(sub, cols, y_col, design=pooled_design_2sch):
    X, names = design(sub, cols)
    y = StandardScaler().fit_transform(sub[[y_col]].values).ravel()
    fit = fit_ols(X, y); fit["cols"] = names
    return fit


def beta_of(fit, name):
    i = fit["cols"].index(name)
    return float(fit["beta"][i]), float(fit["se"][i])


def partial_r(y, x, controls):
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c) for c in controls])
    bx, *_ = np.linalg.lstsq(X, x, rcond=None)
    by, *_ = np.linalg.lstsq(X, y, rcond=None)
    return stats.pearsonr(x - X @ bx, y - X @ by)


def vif_in_design(target_col, other_cols, frame, design):
    """VIF computed inside the actual pooled design (scheme dummies included)."""
    X_oth, _ = design(frame, other_cols)
    z_tgt = StandardScaler().fit_transform(frame[[target_col]].values).ravel()
    beta, *_ = np.linalg.lstsq(X_oth, z_tgt, rcond=None)
    resid = z_tgt - X_oth @ beta
    rss = float(resid @ resid); tss = float(((z_tgt - z_tgt.mean())**2).sum())
    r2 = 1 - rss / tss
    return 1.0 / (1.0 - r2) if r2 < 1 else np.inf


print("Utilities loaded.")

Utilities loaded.


## §1 — Identifiability check first

Before reading any β_cos, check the VIF for cos in the four-control pooled
design (mean_single + max_single + mechanical_push + scheme dummy). If
mechanical_push is essentially redundant with cos under the two-scheme design,
report the model without mechanical_push as the trustworthy one.

In [3]:
# VIF of cos vs other predictors, in both designs
print("=== VIF for cos (inside the actual pooled design) ===\n")
preds = ["mean_single_abs", "max_single_abs", "mechanical_push"]

vif_2sch = vif_in_design("cos", preds, df, pooled_design_2sch)
vif_3sch = vif_in_design("cos", preds, df_all, pooled_design_3sch)
vif_2sch_no_mech = vif_in_design("cos", ["mean_single_abs", "max_single_abs"], df, pooled_design_2sch)

print(f"  2-scheme (v2+v3), {{mean_single, max_single, mech_push, d_per_axis}}:  VIF(cos) = {vif_2sch:8.3f}")
print(f"  3-scheme (n=76),  {{mean_single, max_single, mech_push, d_nT, d_pa}}: VIF(cos) = {vif_3sch:8.3f}")
print(f"  2-scheme (v2+v3) WITHOUT mechanical_push:                           VIF(cos) = {vif_2sch_no_mech:8.3f}")
print()
print("Rule of thumb: VIF > 4 → meaningful collinearity, > 10 → severe.")
print("If the 2-scheme VIF blows up, lean on the no-mech sensitivity model in §2 / §3.")

=== VIF for cos (inside the actual pooled design) ===

  2-scheme (v2+v3), {mean_single, max_single, mech_push, d_per_axis}:  VIF(cos) =    3.889
  3-scheme (n=76),  {mean_single, max_single, mech_push, d_nT, d_pa}: VIF(cos) =    3.927
  2-scheme (v2+v3) WITHOUT mechanical_push:                           VIF(cos) =    1.786

Rule of thumb: VIF > 4 → meaningful collinearity, > 10 → severe.
If the 2-scheme VIF blows up, lean on the no-mech sensitivity model in §2 / §3.


## §2 — Decomposition table (β_cos for magnitude and direction)

Two outcomes, two specifications each (with and without `mechanical_push`).
For each: standardized β_cos, SE, parametric t-p, permutation p (shuffle cos
within scheme, 10000×), bootstrap 95% CI, bivariate r(cos, outcome).

In [4]:
# Headline tables
PREDS_FULL = ["mean_single_abs", "max_single_abs", "mechanical_push", "cos"]
PREDS_NOMECH = ["mean_single_abs", "max_single_abs", "cos"]


def beta_cos_for(frame, y_col, preds, design):
    fit = fit_pooled(frame, preds, y_col, design=design)
    return beta_of(fit, "cos")[0]


def perm_within_scheme(frame, y_col, preds, design, n_perm=10_000, rng=None):
    rng = rng or RNG
    obs = beta_cos_for(frame, y_col, preds, design)
    null = np.empty(n_perm)
    f_loc = frame.copy()
    for i in range(n_perm):
        cos_arr = f_loc["cos"].values.copy()
        for sch in f_loc["scheme"].unique():
            mask = (f_loc["scheme"] == sch).values
            blk = cos_arr[mask].copy(); rng.shuffle(blk); cos_arr[mask] = blk
        f_loc["cos"] = cos_arr
        null[i] = beta_cos_for(f_loc, y_col, preds, design)
    return obs, float(np.mean(np.abs(null) >= np.abs(obs)))


def boot_cos(frame, y_col, preds, design, n_boot=5_000, rng=None):
    rng = rng or RNG
    n = len(frame); out = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        out[i] = beta_cos_for(frame.iloc[idx], y_col, preds, design)
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))


def fit_row(frame, y_col, preds, design, n_perm=10_000, n_boot=5_000):
    fit = fit_pooled(frame, preds, y_col, design=design)
    b, s = beta_of(fit, "cos")
    t = b / s if s > 0 else np.nan
    p_t = 2 * (1 - stats.t.cdf(abs(t), fit["dof"]))
    r_biv, p_biv = stats.pearsonr(frame["cos"], frame[y_col])
    _, p_perm = perm_within_scheme(frame, y_col, preds, design, n_perm=n_perm)
    lo, hi = boot_cos(frame, y_col, preds, design, n_boot=n_boot)
    return {"n": fit["n"], "R²": fit["r2"], "β_cos (std)": b, "SE": s,
            "p_param": p_t, "perm p": p_perm,
            "boot 95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
            "r(cos, y) biv": r_biv, "p biv": p_biv}


print("=== v2+v3 decomposition (n=50, 2-scheme design) ===\n")
rows = []
for label, (y_col, preds) in {
    "MAGNITUDE — mean_joint_abs   (full, w/ mech_push)":  ("mean_joint_abs",   PREDS_FULL),
    "MAGNITUDE — mean_joint_abs   (no mech_push)":         ("mean_joint_abs",   PREDS_NOMECH),
    "DIRECTION — supp_mean_signed (full, w/ mech_push)":   ("supp_mean_signed", PREDS_FULL),
    "DIRECTION — supp_mean_signed (no mech_push)":          ("supp_mean_signed", PREDS_NOMECH),
}.items():
    r = fit_row(df, y_col, preds, pooled_design_2sch)
    r["spec"] = label
    rows.append(r)

dec_v2v3 = pd.DataFrame(rows)[["spec","n","R²","β_cos (std)","SE","p_param","perm p","boot 95% CI","r(cos, y) biv","p biv"]]
print(dec_v2v3.round(4).to_string(index=False))

=== v2+v3 decomposition (n=50, 2-scheme design) ===



                                             spec  n     R²  β_cos (std)     SE  p_param  perm p      boot 95% CI  r(cos, y) biv  p biv
MAGNITUDE — mean_joint_abs   (full, w/ mech_push) 50 0.7756       0.1011 0.1408   0.4765  0.1763 [-0.243, +0.313]         0.6469 0.0000
      MAGNITUDE — mean_joint_abs   (no mech_push) 50 0.7725       0.1813 0.0950   0.0627  0.0149 [-0.013, +0.350]         0.6469 0.0000
DIRECTION — supp_mean_signed (full, w/ mech_push) 50 0.4784       0.4016 0.2147   0.0681  0.0000 [-0.059, +0.755]         0.4824 0.0004
      DIRECTION — supp_mean_signed (no mech_push) 50 0.4759       0.3281 0.1442   0.0277  0.0031 [+0.049, +0.561]         0.4824 0.0004


## §3 — Robustness on the directional outcome (v2+v3)

Same four probes as the directional reframe: + sem_sim, |cos| instead of
signed, leave-one-trait-out, drop apathetic + hallucinating together.

For LOO and drop-both we use the no-mech sensitivity model when VIF is severe
(see §1) — otherwise the full model. The LOO range is what tells us whether
the effect is genuinely pairwise.

In [5]:
# Robustness — directional outcome only
PRIMARY = "supp_mean_signed"
USE_FULL = True   # set False to use no-mech sensitivity; default to full and we'll report both for LOO

# 3a. + sem_sim
preds_with_sem = ["mean_single_abs", "max_single_abs", "mechanical_push", "sem_sim", "cos"]
fit_sem = fit_pooled(df, preds_with_sem, PRIMARY, design=pooled_design_2sch)
b_sem, s_sem = beta_of(fit_sem, "cos")
t_sem = b_sem / s_sem if s_sem > 0 else np.nan
p_sem = 2 * (1 - stats.t.cdf(abs(t_sem), fit_sem["dof"]))
# perm p with sem_sim in
_, p_perm_sem = perm_within_scheme(df, PRIMARY, preds_with_sem, pooled_design_2sch)
b_sem_sim, s_sem_sim = beta_of(fit_sem, "sem_sim")
print(f"=== 3a. + sem_sim (directional, v2+v3) ===")
print(f"  β_cos = {b_sem:+.4f}  SE = {s_sem:.4f}  param p = {p_sem:.4f}  perm p = {p_perm_sem:.4f}")
print(f"  β_sem_sim = {b_sem_sim:+.4f}  SE = {s_sem_sim:.4f}")
print()

# 3b. |cos| instead of signed cos — bivariate (the partial fit needs a re-engineered control set;
# bivariate r is what we actually care about for the sign-mattering check)
print("=== 3b. signed cos vs |cos| on directional outcome ===")
for sch in [*sorted(df['scheme'].unique()), 'pooled']:
    sub = df if sch == 'pooled' else df[df['scheme'] == sch]
    rs, ps = stats.pearsonr(sub['cos'], sub[PRIMARY])
    ra, pa = stats.pearsonr(sub['cos'].abs(), sub[PRIMARY])
    print(f"  {sch:10s} n={len(sub):2d}  r(signed) = {rs:+.3f} (p={ps:.4f})   r(|cos|) = {ra:+.3f} (p={pa:.4f})")

=== 3a. + sem_sim (directional, v2+v3) ===
  β_cos = +0.4831  SE = 0.2267  param p = 0.0389  perm p = 0.0000
  β_sem_sim = -0.1274  SE = 0.1161

=== 3b. signed cos vs |cos| on directional outcome ===
  normTrue   n=28  r(signed) = +0.475 (p=0.0106)   r(|cos|) = +0.021 (p=0.9173)
  per_axis   n=22  r(signed) = +0.472 (p=0.0264)   r(|cos|) = +0.233 (p=0.2972)
  pooled     n=50  r(signed) = +0.482 (p=0.0004)   r(|cos|) = +0.117 (p=0.4203)


In [6]:
# 3c. Leave-one-trait-out — both with and without mechanical_push
TRAITS = sorted(set(df["trait_a"]).union(df["trait_b"]))
print(f"Traits in v2+v3 frame: {TRAITS}\n")


def d2_beta_cos(frame, preds, y_col=PRIMARY):
    fit = fit_pooled(frame, preds, y_col, design=pooled_design_2sch)
    b, s = beta_of(fit, "cos")
    t = b / s if s > 0 else np.nan
    p = 2 * (1 - stats.t.cdf(abs(t), fit["dof"]))
    return b, s, p, fit["r2"], fit["n"]


def loo_table(preds, label):
    b0, s0, p0, r2_0, n0 = d2_beta_cos(df, preds)
    rows = [{"drop_trait": "(none — full)", "n": n0, "β_cos": b0,
             "SE": s0, "p_param": p0, "R²": r2_0}]
    for tr in TRAITS:
        sub = df[~((df["trait_a"] == tr) | (df["trait_b"] == tr))]
        if sub["scheme"].nunique() < 2 or len(sub) < len(preds) + 3:
            rows.append({"drop_trait": tr, "n": len(sub), "β_cos": np.nan,
                         "SE": np.nan, "p_param": np.nan, "R²": np.nan})
            continue
        b, s, p, r2, n = d2_beta_cos(sub, preds)
        rows.append({"drop_trait": tr, "n": n, "β_cos": b, "SE": s, "p_param": p, "R²": r2})
    loo = pd.DataFrame(rows)
    print(f"=== 3c-{label}. LOO with predictors = {preds} ===\n")
    print(loo.round(3).to_string(index=False))
    bvals = loo["β_cos"].dropna()
    print(f"\n  β_cos range across drops: [{bvals.min():+.3f}, {bvals.max():+.3f}]")
    print(f"  β_cos full = {b0:+.3f};  median drop = {bvals.median():+.3f}\n")
    return loo


loo_full   = loo_table(PREDS_FULL,   "full")
loo_nomech = loo_table(PREDS_NOMECH, "nomech")

Traits in v2+v3 frame: ['apathetic', 'confidence', 'evil', 'formality', 'hallucinating', 'humorous', 'impolite', 'sycophantic']

=== 3c-full. LOO with predictors = ['mean_single_abs', 'max_single_abs', 'mechanical_push', 'cos'] ===

   drop_trait  n  β_cos    SE  p_param    R²
(none — full) 50  0.402 0.215    0.068 0.478
    apathetic 37  0.165 0.296    0.582 0.560
   confidence 37  0.516 0.228    0.031 0.615
         evil 37  0.387 0.231    0.104 0.543
    formality 37  0.427 0.273    0.127 0.347
hallucinating 39  0.605 0.243    0.018 0.496
     humorous 40  0.426 0.212    0.052 0.476
     impolite 37  0.294 0.324    0.372 0.422
  sycophantic 36  0.418 0.246    0.100 0.558

  β_cos range across drops: [+0.165, +0.605]
  β_cos full = +0.402;  median drop = +0.418

=== 3c-nomech. LOO with predictors = ['mean_single_abs', 'max_single_abs', 'cos'] ===

   drop_trait  n  β_cos    SE  p_param    R²
(none — full) 50  0.328 0.144    0.028 0.476
    apathetic 37  0.106 0.200    0.602 0.559
   

In [7]:
# 3d. Drop apathetic + hallucinating together
DROP_BOTH = ["apathetic", "hallucinating"]
mask = ~((df["trait_a"].isin(DROP_BOTH)) | (df["trait_b"].isin(DROP_BOTH)))
sub = df[mask]

print("=== 3d. Drop apathetic + hallucinating together (directional, v2+v3) ===\n")
for label, preds in [("full (w/ mech_push)", PREDS_FULL),
                     ("no mech_push", PREDS_NOMECH)]:
    b_full, s_full, p_full, r2_full, n_full = d2_beta_cos(df, preds)
    b, s, p, r2, n = d2_beta_cos(sub, preds)
    _, p_perm_drop = perm_within_scheme(sub, PRIMARY, preds, pooled_design_2sch)
    print(f"  {label}")
    print(f"    full       n={n_full}  β_cos = {b_full:+.4f}  SE={s_full:.4f}  p={p_full:.4f}  R²={r2_full:.3f}")
    print(f"    drop both  n={n:2d}      β_cos = {b:+.4f}  SE={s:.4f}  p={p:.4f}  R²={r2:.3f}")
    print(f"    drop both perm p = {p_perm_drop:.4f}\n")

=== 3d. Drop apathetic + hallucinating together (directional, v2+v3) ===



  full (w/ mech_push)
    full       n=50  β_cos = +0.4016  SE=0.2147  p=0.0681  R²=0.478
    drop both  n=27      β_cos = +0.7055  SE=0.3559  p=0.0607  R²=0.631
    drop both perm p = 0.0000



  no mech_push
    full       n=50  β_cos = +0.3281  SE=0.1442  p=0.0277  R²=0.476
    drop both  n=27      β_cos = +0.5757  SE=0.2566  p=0.0353  R²=0.626
    drop both perm p = 0.0001



## §4 — Side-by-side comparison: n=76 (all 3 schemes) vs n=50 (v2 + v3)

For both outcomes, in the matched full specification (the one each frame
naturally identifies):
- n=76: three schemes, two dummies, full predictor set including
  mechanical_push.
- n=50: two schemes, one dummy. Reported in two variants — *with*
  mechanical_push (matching the n=76 specification but expect VIF inflation)
  and *without* mechanical_push (the trustworthy sensitivity if VIF is
  severe, per §1).

In [8]:
# Compute n=76 (three-scheme) reference using the same fit_row plumbing
def fit_row_3sch(frame, y_col, preds, n_perm=10_000, n_boot=5_000):
    return fit_row(frame, y_col, preds, pooled_design_3sch, n_perm=n_perm, n_boot=n_boot)


print("=== SIDE BY SIDE: n=76 vs n=50 ===\n")
all_rows = []
for outcome_label, y_col in [("MAGNITUDE  (mean_joint_abs)",   "mean_joint_abs"),
                              ("DIRECTION  (supp_mean_signed)", "supp_mean_signed")]:
    print(f"--- {outcome_label} ---")
    rows = []
    # n=76, full predictor set with mech_push
    r76 = fit_row_3sch(df_all, y_col, PREDS_FULL)
    r76["frame"] = "n=76 (3-scheme, with mech_push)"
    rows.append(r76)
    # n=50, with mech_push
    r50f = fit_row(df, y_col, PREDS_FULL, pooled_design_2sch)
    r50f["frame"] = "n=50 (2-scheme, with mech_push)"
    rows.append(r50f)
    # n=50, no mech_push
    r50n = fit_row(df, y_col, PREDS_NOMECH, pooled_design_2sch)
    r50n["frame"] = "n=50 (2-scheme, NO mech_push)"
    rows.append(r50n)

    tbl = pd.DataFrame(rows)[["frame","n","R²","β_cos (std)","SE",
                              "perm p","boot 95% CI","r(cos, y) biv","p biv"]]
    print(tbl.round(4).to_string(index=False))
    print()
    all_rows.extend(rows)

=== SIDE BY SIDE: n=76 vs n=50 ===

--- MAGNITUDE  (mean_joint_abs) ---


                          frame  n     R²  β_cos (std)     SE  perm p      boot 95% CI  r(cos, y) biv  p biv
n=76 (3-scheme, with mech_push) 76 0.6987       0.0515 0.1309  0.4430 [-0.230, +0.254]         0.5170    0.0
n=50 (2-scheme, with mech_push) 50 0.7756       0.1011 0.1408  0.1708 [-0.240, +0.313]         0.6469    0.0
  n=50 (2-scheme, NO mech_push) 50 0.7725       0.1813 0.0950  0.0140 [-0.019, +0.351]         0.6469    0.0

--- DIRECTION  (supp_mean_signed) ---


                          frame  n     R²  β_cos (std)     SE  perm p      boot 95% CI  r(cos, y) biv  p biv
n=76 (3-scheme, with mech_push) 76 0.4459       0.2345 0.1776  0.0085 [-0.122, +0.529]         0.3721 0.0009
n=50 (2-scheme, with mech_push) 50 0.4784       0.4016 0.2147  0.0005 [-0.067, +0.752]         0.4824 0.0004
  n=50 (2-scheme, NO mech_push) 50 0.4759       0.3281 0.1442  0.0026 [+0.051, +0.566]         0.4824 0.0004



## §5 — Final printable summary

Re-prints the key numbers in a clean block ready to paste.

In [9]:
print("="*72)
print("FINAL NUMBERS  (v2 + v3 only, coh ≥ 30, n =", len(df), ")")
print("="*72)
print()
print(f"per-scheme: {dict(df['scheme'].value_counts())}")
print(f"VIF(cos) in 2-scheme {{mean_single, max_single, mech_push, d_per_axis}}: {vif_2sch:.2f}")
print(f"VIF(cos) in 2-scheme without mech_push:                                  {vif_2sch_no_mech:.2f}")
trustworthy = "no mech_push" if vif_2sch > 4 else "full (w/ mech_push)"
print(f"trustworthy spec under VIF rule of thumb (>4): {trustworthy}")
print()

# Pull the four cells we want from the side-by-side tables
def grab(rows, frame_match, key):
    for r in rows:
        if r["frame"] == frame_match:
            return r[key]
    return None


print("STANDARDIZED β_cos  (with permutation p in parentheses)")
print()
print(f"  MAGNITUDE")
print(f"    n=76 (3-sch, w/ mech):   β = {grab(all_rows,'n=76 (3-scheme, with mech_push)','β_cos (std)'):+.4f}   perm p = {grab(all_rows,'n=76 (3-scheme, with mech_push)','perm p'):.4f}   boot CI {grab(all_rows,'n=76 (3-scheme, with mech_push)','boot 95% CI')}")
print(f"    n=50 (2-sch, w/ mech):   β = {grab(all_rows,'n=50 (2-scheme, with mech_push)','β_cos (std)'):+.4f}   perm p = {grab(all_rows,'n=50 (2-scheme, with mech_push)','perm p'):.4f}   boot CI {grab(all_rows,'n=50 (2-scheme, with mech_push)','boot 95% CI')}")
print(f"    n=50 (2-sch, no mech):   β = {grab(all_rows,'n=50 (2-scheme, NO mech_push)','β_cos (std)'):+.4f}   perm p = {grab(all_rows,'n=50 (2-scheme, NO mech_push)','perm p'):.4f}   boot CI {grab(all_rows,'n=50 (2-scheme, NO mech_push)','boot 95% CI')}")
print()
print(f"  DIRECTION")
# slice to direction rows (rows[3:6])
dir_rows = [r for r in all_rows if "frame" in r and r in all_rows[3:]]
print(f"    n=76 (3-sch, w/ mech):   β = {all_rows[3]['β_cos (std)']:+.4f}   perm p = {all_rows[3]['perm p']:.4f}   boot CI {all_rows[3]['boot 95% CI']}")
print(f"    n=50 (2-sch, w/ mech):   β = {all_rows[4]['β_cos (std)']:+.4f}   perm p = {all_rows[4]['perm p']:.4f}   boot CI {all_rows[4]['boot 95% CI']}")
print(f"    n=50 (2-sch, no mech):   β = {all_rows[5]['β_cos (std)']:+.4f}   perm p = {all_rows[5]['perm p']:.4f}   boot CI {all_rows[5]['boot 95% CI']}")
print()

# Direction robustness summary
print("DIRECTIONAL ROBUSTNESS (v2+v3 frame)")
print(f"  + sem_sim:                β_cos = {b_sem:+.4f}  perm p = {p_perm_sem:.4f}")
print(f"  drop apath+halluc (full): β_cos = {d2_beta_cos(sub, PREDS_FULL)[0]:+.4f}  (n={d2_beta_cos(sub, PREDS_FULL)[4]})")
print(f"  drop apath+halluc (no mech): β_cos = {d2_beta_cos(sub, PREDS_NOMECH)[0]:+.4f}")
print(f"  LOO β_cos range (full):   [{loo_full['β_cos'].dropna().min():+.3f}, {loo_full['β_cos'].dropna().max():+.3f}]")
print(f"  LOO β_cos range (no mech):[{loo_nomech['β_cos'].dropna().min():+.3f}, {loo_nomech['β_cos'].dropna().max():+.3f}]")
print()
print("="*72)

FINAL NUMBERS  (v2 + v3 only, coh ≥ 30, n = 50 )

per-scheme: {'normTrue': 28, 'per_axis': 22}
VIF(cos) in 2-scheme {mean_single, max_single, mech_push, d_per_axis}: 3.89
VIF(cos) in 2-scheme without mech_push:                                  1.79
trustworthy spec under VIF rule of thumb (>4): full (w/ mech_push)

STANDARDIZED β_cos  (with permutation p in parentheses)

  MAGNITUDE
    n=76 (3-sch, w/ mech):   β = +0.0515   perm p = 0.4430   boot CI [-0.230, +0.254]
    n=50 (2-sch, w/ mech):   β = +0.1011   perm p = 0.1708   boot CI [-0.240, +0.313]
    n=50 (2-sch, no mech):   β = +0.1813   perm p = 0.0140   boot CI [-0.019, +0.351]

  DIRECTION
    n=76 (3-sch, w/ mech):   β = +0.2345   perm p = 0.0085   boot CI [-0.122, +0.529]
    n=50 (2-sch, w/ mech):   β = +0.4016   perm p = 0.0005   boot CI [-0.067, +0.752]
    n=50 (2-sch, no mech):   β = +0.3281   perm p = 0.0026   boot CI [+0.051, +0.566]

DIRECTIONAL ROBUSTNESS (v2+v3 frame)
  + sem_sim:                β_cos = +0.4831  pe